# CO 1 Notes - Real-Time Networks Communication and Security for Autonomous Systems

These notes cover the CO 1 foundation topics for real-time systems and real-time communication in autonomous systems. The notebook is written as textbook-style study material with definitions, architectures, formulas, examples, and case studies.

The code cells are small executable demonstrations. They are not a replacement for certified real-time analysis tools; they are included to make the timing formulas concrete.

## 1. Introduction to Real-Time Systems

A real-time system is a computing system in which correctness depends on both the value produced and the time at which the value is produced. A result that is logically correct but delivered late can still be wrong for the system.

Logical correctness means the output value is correct. Temporal correctness means the output is produced before the required deadline. In autonomous systems, both are required. For example, an emergency braking controller must calculate the correct braking decision and deliver it before the vehicle loses the safe stopping window.

Real-time systems are different from general-purpose systems. A general-purpose desktop or server system usually optimizes average response time, user convenience, throughput, and fairness. A real-time system optimizes bounded response time, deadline satisfaction, deterministic behavior, and predictable worst-case performance.

Real-time does not mean merely fast. A fast system with unpredictable delay spikes is weak for hard real-time control. A slower system with guaranteed upper bounds may be better if it consistently meets deadlines.

### Key Properties

Determinism means the system behavior is repeatable under defined conditions. Predictability means the designer can reason about the timing behavior before deployment. Bounded latency means delay has a known upper limit. Temporal correctness means the result is available while it is still useful.

### Basic Architecture

```
Physical Event
  -> Sensor / Input Interface
  -> Real-Time Task Released
  -> Scheduler
  -> Processor / Communication Medium
  -> Output Decision
  -> Actuator / Receiver
  -> Deadline Verification
```

The architecture shows why real-time behavior is an end-to-end property. A fast processor alone cannot guarantee real-time correctness if sensing, queueing, communication, or actuation is delayed.

## Complete CO 1 Textbook Notes

### Meaning of a Real-Time System

A real-time system is a system where time is part of correctness. The output must be logically correct and temporally correct. Logical correctness asks whether the computed value, command, or message is right. Temporal correctness asks whether the value is produced before its deadline. If either part fails, the real-time result is not acceptable.

In general-purpose computing, the main performance goals are usually average speed, user experience, resource utilization, and throughput. In real-time computing, the main goals are bounded response time, predictable scheduling, bounded communication delay, and deadline satisfaction. This is why a real-time system is not defined by being very fast. It is defined by being timely under specified conditions.

The design of a real-time system begins with identifying events, tasks, timing parameters, and constraints. A physical event may be an obstacle detection, wheel-speed update, steering-angle change, packet arrival, temperature threshold, or user command. The software task created by that event must be scheduled, executed, and completed before the result becomes invalid or unsafe.

### Real-Time System Correctness

Correctness can be written as:

```
real_time_correctness = logical_correctness AND temporal_correctness
```

Logical correctness examples:

```
detected_object = true
computed_brake_force = valid
received_message_payload = valid
```

Temporal correctness examples:

```
finish_time <= absolute_deadline
communication_delay <= message_deadline
verification_time <= security_decision_deadline
```

In autonomous systems, temporal correctness must be evaluated end to end. A perception module may detect an obstacle correctly, but if the braking command reaches the actuator too late, the system has still failed its real-time requirement.

### Determinism and Predictability

Determinism means the system behaves in a repeatable way under defined inputs and conditions. Predictability means engineers can analyze and bound the timing behavior before deployment. These properties are critical because safety decisions must not depend on random delay spikes, uncontrolled queueing, or unpredictable scheduling.

Deterministic behavior can be improved through fixed-priority scheduling, time-triggered scheduling, bounded interrupt handling, memory allocation control, traffic shaping, priority message queues, synchronized clocks, and deterministic network technologies. However, determinism always depends on assumptions. If workload, network traffic, interrupts, or execution paths are not bounded, the timing guarantee is weak.

### Hard, Firm, and Soft Real-Time Systems

Hard real-time systems must meet deadlines. A missed deadline may cause unsafe operation or system failure. Examples include airbag control, autonomous emergency braking, anti-lock braking control, industrial emergency stop, flight control, medical infusion control, and protection relays.

Firm real-time systems allow occasional misses, but late results have no value. The result is discarded. Examples include stale object-detection frames, expired traffic-signal messages, late cooperative-awareness messages, and sensor-fusion inputs that arrive after the fusion window closes.

Soft real-time systems allow late results with degraded quality. Examples include dashboard display refresh, passenger infotainment, non-critical telemetry upload, map rendering, cabin comfort control, and general monitoring dashboards.

The correct classification depends on the consequence of lateness. The same sensor can support hard, firm, or soft functions depending on how its output is used.

### Task and Event Types

Periodic tasks repeat at fixed intervals:

```
Task_i = (C_i, T_i, D_i)
```

where `C_i` is execution time, `T_i` is period, and `D_i` is relative deadline. Periodic tasks appear in sensor sampling, controller update loops, heartbeat messages, and actuator supervision.

Aperiodic tasks occur irregularly and do not have a guaranteed minimum inter-arrival time. Examples include diagnostic requests, user commands, or maintenance queries. Aperiodic tasks can overload the system if admitted without control.

Sporadic tasks occur irregularly but have a known minimum inter-arrival time. Examples include emergency alerts that are unpredictable but physically bounded. Sporadic modelling is valuable because it supports worst-case analysis.

Time-triggered events are released by a clock. Event-triggered events are released by conditions such as packet arrival, obstacle detection, threshold crossing, or interrupt. Time-triggered systems are easier to analyze. Event-triggered systems react quickly but need overload protection.

### Timing Parameters

The main timing parameters are event occurrence time, release time, arrival time, start time, execution time, waiting time, completion time, response time, and turnaround time.

```
waiting_time = start_time - release_time
finish_time = start_time + execution_time
response_time = finish_time - release_time
turnaround_time = finish_time - arrival_time
```

Release time and arrival time are sometimes treated as the same in simple CPU scheduling problems. In communication systems they can be different. A real-world event may occur before the software receives a packet. A packet may arrive at the network interface before the application task is released.

### Deadlines, Slack, and Laxity

Relative deadline is measured from task release. Absolute deadline is a point on the timeline:

```
absolute_deadline = release_time + relative_deadline
```

Deadline status:

```
deadline_met = finish_time <= absolute_deadline
deadline_missed = finish_time > absolute_deadline
```

Deadline margin:

```
deadline_margin = absolute_deadline - finish_time
```

Slack and laxity estimate available spare time:

```
slack = absolute_deadline - current_time - remaining_execution_time
laxity = deadline - current_time - remaining_computation_time
```

If slack is negative, the task cannot meet its deadline unless assumptions change. If slack is small, the system is fragile because small delays can cause a miss.

### WCET and BCET

Worst-Case Execution Time is the maximum execution time under stated assumptions. Best-Case Execution Time is the minimum. Average execution time is not enough for hard real-time systems. A controller that usually finishes in 2 ms but sometimes takes 30 ms cannot be accepted for a 10 ms hard deadline unless the 30 ms path is eliminated or bounded.

WCET depends on algorithm paths, processor architecture, cache behavior, interrupts, memory access, compiler settings, input size, and operating-system behavior. In classroom simulations, execution time is usually assumed. In real engineering, it must be measured, bounded, or formally analyzed.

### Communication Performance Parameters

Latency is the time from send to receive:

```
latency = receive_time - send_time
```

Jitter is variation in latency:

```
jitter_i = abs(latency_i - latency_(i-1))
```

Throughput is useful delivered data per observation time:

```
throughput = delivered_bits / observation_time
```

Bandwidth is nominal link capacity. Throughput is actual useful delivery after overhead, queueing, loss, contention, retransmission, encryption metadata, acknowledgements, and routing.

Packet transmission time:

```
transmission_time = packet_size_bits / link_rate_bits_per_second
```

End-to-end delay:

```
end_to_end_delay = processing_delay + queueing_delay + transmission_delay + propagation_delay
```

Reliability:

```
packet_loss_rate = lost_packets / sent_packets
delivery_ratio = delivered_packets / sent_packets
```

### Real-Time Communication Requirements

Real-time communication requires bounded latency, low jitter, predictable message delivery, reliability, availability, and deadline-aware scheduling. It is not enough for a network to have high bandwidth. A high-bandwidth network with unpredictable queueing may still be poor for control.

Deadline-aware communication prioritizes messages by urgency and usefulness. For example, an emergency braking alert should not wait behind bulk telemetry. Deterministic communication can use time slots, priority arbitration, traffic shaping, admission control, redundancy, and synchronized clocks.

### Comparison Tables for Revision

| Concept | Meaning | Typical Mistake |
|---|---|---|
| Latency | Delay from sender to receiver | Confusing it with throughput |
| Jitter | Variation in delay | Reporting only average latency |
| Bandwidth | Nominal link capacity | Assuming full bandwidth is usable payload |
| Throughput | Actual useful delivered data rate | Ignoring packet loss and protocol overhead |
| WCET | Maximum execution time under assumptions | Using average time for hard real-time claims |
| Slack | Spare time before deadline after remaining work | Treating small positive slack as always safe |

| System Type | Main Question | Correct Lab Interpretation |
|---|---|---|
| Hard real-time | Was every deadline met? | Any miss is unacceptable under the model |
| Firm real-time | Was the result still useful? | Late result is discarded |
| Soft real-time | How much quality degraded? | Misses reduce service quality but may not fail the system |

| Trigger Type | Advantage | Risk |
|---|---|---|
| Time-triggered | Predictable schedule | Less responsive to unexpected events |
| Event-triggered | Fast reaction to external events | Burst arrivals can overload queues |
| Sporadic | Supports analysis with minimum inter-arrival time | Incorrect minimum-spacing assumption invalidates analysis |

### Autonomous-System Timing Analysis

Autonomous-system timing must be evaluated from sensing to physical action:

```
sensor_capture
  + preprocessing
  + perception
  + sensor_fusion
  + planning
  + control
  + communication
  + actuator_response
  = perception_to_action_delay
```

For braking:

```
reaction_distance = vehicle_speed * total_system_delay
braking_distance = vehicle_speed^2 / (2 * deceleration)
stopping_distance = reaction_distance + braking_distance
```

The safe condition is:

```
stopping_distance <= obstacle_distance
```

This formula shows why timing is a safety variable. Increasing delay increases reaction distance, which reduces safety margin.

## Architectures, Block Diagrams, and Flowcharts

### A. Generic Real-Time System Architecture

```
+------------------+      +------------------+      +------------------+
| Physical Process | ---> | Sensor Interface | ---> | Real-Time Task   |
+------------------+      +------------------+      +------------------+
                                                           |
                                                           v
                         +------------------+      +------------------+
                         | Deadline Table   | <--- | Scheduler        |
                         +------------------+      +------------------+
                                                           |
                                                           v
+------------------+      +------------------+      +------------------+
| Physical Action  | <--- | Actuator Driver | <--- | Control Output   |
+------------------+      +------------------+      +------------------+
```

Explanation: The physical process produces events. Sensors convert physical values to digital inputs. Tasks process the inputs. The scheduler controls execution order. The deadline table defines timing constraints. The actuator driver converts the software output into physical action.

### B. Layered Architecture for Autonomous Real-Time Systems

```
+-------------------------------------------------------------+
| Application Layer: braking, steering, navigation, warning    |
+-------------------------------------------------------------+
| Decision Layer: planning, control, rule engine, IDS action   |
+-------------------------------------------------------------+
| Perception Layer: object detection, tracking, fusion         |
+-------------------------------------------------------------+
| Communication Layer: CAN, Ethernet, Wi-Fi, LTE, 5G, V2X      |
+-------------------------------------------------------------+
| Platform Layer: RTOS, scheduler, clock, memory, drivers      |
+-------------------------------------------------------------+
| Hardware Layer: sensors, ECU, edge unit, actuator, vehicle   |
+-------------------------------------------------------------+
```

Timing can be lost at any layer. Security checks may sit in the communication and decision layers. A complete analysis must include both computation delay and communication delay.

### C. Real-Time Task Life Cycle Flowchart

```
        +----------------+
        | Event Occurs   |
        +-------+--------+
                |
                v
        +----------------+
        | Task Released  |
        +-------+--------+
                |
                v
        +----------------+
        | Ready Queue    |
        +-------+--------+
                |
                v
        +----------------+
        | Scheduler Picks|
        +-------+--------+
                |
                v
        +----------------+
        | Task Executes  |
        +-------+--------+
                |
                v
        +----------------+
        | Task Finishes  |
        +-------+--------+
                |
                v
        +----------------------------+
        | finish <= absolute deadline?|
        +----------+-----------------+
                   |
          +--------+--------+
          |                 |
          v                 v
 +----------------+  +----------------+
 | Accept Result  |  | Miss Handling  |
 +----------------+  +----------------+
```

Miss handling depends on task class. Hard real-time miss means failure handling. Firm real-time miss means discard. Soft real-time miss means degraded service.

### D. Deadline Verification Flowchart

```
Start
  |
  v
Read release time, execution time, deadline
  |
  v
Compute absolute_deadline = release + relative_deadline
  |
  v
Compute finish_time = start + execution
  |
  v
Compute margin = absolute_deadline - finish_time
  |
  v
Is margin >= 0?
  | Yes                         | No
  v                             v
Deadline met              Deadline missed
  |                             |
  v                             v
Record accepted result     Apply hard/firm/soft miss policy
```

### E. Real-Time Communication Block Diagram

```
+-----------+    +------------+    +------------+    +------------+
| Sender    | -> | Tx Queue   | -> | Network    | -> | Rx Queue   |
+-----------+    +------------+    +------------+    +------------+
      |                |                |                 |
      v                v                v                 v
 Send time       Queueing delay   Link delay        Arrival time
                                                         |
                                                         v
                                               +----------------+
                                               | Receiver Task  |
                                               +----------------+
```

The total communication delay includes sender processing, queueing, transmission, propagation, receiver queueing, and receiver processing.

### F. Deadline-Aware Communication Flowchart

```
Message generated
  |
  v
Classify criticality and deadline
  |
  v
Is message safety critical?
  | Yes
  v
Place in high-priority / reserved traffic class
  |
  v
Transmit before non-critical traffic

If No:
  |
  v
Place in normal or background queue
  |
  v
Transmit when bandwidth is available
```

This avoids treating all packets equally when their timing value is not equal.

### G. Autonomous Braking Timing Architecture

```
Obstacle
  -> Camera/Radar/Lidar
  -> Sensor Preprocessing
  -> Object Detection
  -> Tracking and Fusion
  -> Collision-Risk Calculation
  -> Brake Decision
  -> Vehicle Bus Message
  -> Brake ECU
  -> Hydraulic/Electric Actuation
  -> Vehicle Deceleration
```

Timing equation:

```
T_total = T_sensor + T_preprocess + T_detect + T_fusion
        + T_plan + T_control + T_bus + T_actuator
```

Deadline condition:

```
T_total <= T_required_for_safe_braking
```

### H. Secure V2X Timing and Security Architecture

```
Sender Vehicle
  -> Create V2X Message
  -> Add Timestamp / Sequence Number
  -> Integrity Protection / Signature
  -> Wireless Transmission
  -> Receiver Verification
  -> Replay Check
  -> Plausibility Check
  -> Accept / Reject / Alert
```

Security and timing are coupled:

```
message_delay = transmission_delay + verification_delay + IDS_delay
message_valid = authenticated AND fresh AND timely
```

An old but authentic message can still be dangerous. A timely but unauthenticated message can also be dangerous.

### I. Scheduling Architecture: RMS vs EDF

```
RMS:
Task period -> Fixed priority -> Ready queue -> Highest fixed priority runs

EDF:
Task release + relative deadline -> Absolute deadline -> Ready queue -> Earliest deadline runs
```

RMS is simple and predictable because priorities are fixed. EDF can use CPU capacity more flexibly because priorities change with deadlines. Both require bounded execution times and valid assumptions.

### J. Lab-Answer Flowchart

```
Read problem
  |
  v
Identify system type: hard, firm, or soft
  |
  v
List timing parameters and communication parameters
  |
  v
Write formulas with variable definitions
  |
  v
Draw architecture or flowchart
  |
  v
Calculate deadline, latency, jitter, or throughput
  |
  v
Interpret result using case-study consequence
  |
  v
State limitations and assumptions
```

## Expanded Case Studies

### Case Study 1 - Autonomous Emergency Braking

An autonomous vehicle travels at 20 m/s and detects an obstacle. The system must process sensor data, calculate risk, and apply brakes before the vehicle reaches the obstacle. This case is hard real-time because a late result can directly affect safety.

Architecture:

```
Obstacle -> Sensor -> Perception -> Risk Calculation -> Control -> Brake ECU -> Actuator
```

Timing calculation:

```
total_delay = sensing + perception + planning + communication + actuation
reaction_distance = speed * total_delay
braking_distance = speed^2 / (2 * deceleration)
stopping_distance = reaction_distance + braking_distance
```

Interpretation: If stopping distance is less than obstacle distance, the timing budget is acceptable under the simplified model. If stopping distance is greater, the system response is too late. The answer must mention that real vehicles require richer models including road friction, brake condition, slope, sensor uncertainty, and validation testing.

### Case Study 2 - Robotic Arm Safety Stop

A robotic arm works near a human operator. A light curtain or proximity sensor detects intrusion into a safety zone. The controller must stop the arm before it crosses a dangerous boundary.

Flow:

```
Human enters zone -> Sensor interrupt -> Safety task -> Stop command -> Drive shutdown
```

Important timing parameters:

```
event_occurrence_time
release_time
interrupt_latency
control_execution_time
network_delay
drive_shutdown_time
physical_stop_time
```

This is hard real-time if human safety depends on the stop action. The system must be designed for worst-case response, not average response.

### Case Study 3 - V2X Hazard Warning

A vehicle detects sudden braking and sends a V2X warning. Nearby vehicles must receive, verify, and use the message before it becomes stale.

Block diagram:

```
Lead Vehicle -> Hazard Message -> Wireless Channel -> Receiver Verification -> Warning Decision
```

The message must be timely and secure:

```
transmission_delay + verification_delay <= warning_deadline
freshness_valid = timestamp within allowed window
integrity_valid = recomputed_digest matches trusted digest or signature verification passes
```

This case connects latency, jitter, packet loss, authentication, freshness, and deadline-aware communication.

### Case Study 4 - Industrial Conveyor Sorting

A conveyor sensor detects a product. The controller classifies it and activates a diverter at the correct location. If the command is late, the product passes the diverter.

Timing:

```
available_time = distance_to_diverter / conveyor_speed
system_response_time = sensor_delay + classification_time + controller_time + actuator_delay
```

Deadline condition:

```
system_response_time <= available_time
```

This can be firm real-time when the late classification is discarded, or hard real-time if a wrong or late action causes machine damage or safety risk.

### Case Study 5 - CAN Bus Message Priority

Several electronic control units attempt to transmit at the same time. CAN arbitration gives bus access to the frame with the lowest numerical identifier. This supports real-time behavior because urgent messages can be assigned high priority.

Example:

```
Brake message ID     = 0x080
Engine message ID    = 0x120
Window status ID     = 0x300
```

The brake message wins because `0x080` is the lowest numeric identifier. The design lesson is that identifier assignment is a timing and safety decision, not just a naming convention.

### Case Study 6 - Edge Obstacle Detection

A warehouse robot uses an onboard edge processor to detect obstacles. Sending all data to the cloud would add network delay and jitter. Edge computing reduces the perception-to-action delay.

Architecture:

```
Depth Sensor -> Edge Processor -> Obstacle Decision -> Motor Controller -> Stop/Turn
```

The key tradeoff is local resource limitation versus cloud delay. Edge devices may have less compute power than cloud servers, but their communication delay is much lower.

### Case Study 7 - MQTT Fleet Telemetry

Vehicles publish telemetry to MQTT topics. Dashboards and analytics services subscribe to those topics. This is usually soft or firm real-time depending on the use.

Architecture:

```
Vehicle Sensor -> MQTT Publisher -> Broker -> Subscriber -> Dashboard / Alert Engine
```

Telemetry formula:

```
message_rate = number_of_messages / time
payload_throughput = total_payload_bytes / time
```

If telemetry is used only for logging, occasional delay may be acceptable. If telemetry triggers hazard alerts, deadline and reliability requirements become stricter.

### Case Study 8 - Replay and Spoofing Detection

An attacker may replay an old valid message or spoof a false vehicle position. A simple IDS can check timestamps, sequence numbers, duplicate messages, impossible speed, or implausible location changes.

IDS rules:

```
replay_flag = timestamp <= last_seen_timestamp
implied_speed = distance_between_claims / time_difference
spoof_flag = implied_speed > maximum_physical_speed
```

Security conclusion: A rule-based IDS is explainable but incomplete. Production systems require layered controls including authentication, freshness checks, plausibility checks, logging, and incident response.

### Case Study 9 - Risk Analysis for Secure V2X

Risk analysis identifies assets, threats, vulnerabilities, likelihood, impact, controls, and residual risk.

```
Asset -> Threat -> Vulnerability -> Likelihood -> Impact -> Risk Score -> Mitigation
```

Simple scoring:

```
risk_score = likelihood * impact
```

Example: A spoofed roadside unit could send false traffic instructions. The asset is infrastructure trust. The vulnerability is weak certificate validation. The mitigation is certificate-chain validation, trust-anchor management, and rejection of unsigned or stale messages.

### Final Study Checklist

For any CO 1 answer, include:

1. Definition with logical and temporal correctness.
2. Classification as hard, firm, or soft.
3. Task type: periodic, aperiodic, or sporadic.
4. Timing parameters and formulas.
5. Communication metrics if networking is involved.
6. Architecture or flowchart.
7. Case-study interpretation.
8. Assumptions and limitations.
9. References to standards or official documentation where applicable.

## 2. Classification of Real-Time Systems

Real-time systems are commonly classified into hard, firm, and soft real-time systems.

| Type | Deadline Meaning | Late Result | Examples |
|---|---|---|---|
| Hard real-time | Deadline must not be missed | System failure or unsafe condition | Airbag trigger, emergency braking, flight control, industrial safety shutdown |
| Firm real-time | Occasional misses may occur, but late result has no value | Result is discarded | Stale object-detection frame, expired V2X warning, late traffic-sign recognition |
| Soft real-time | Deadline miss degrades quality | Result still has reduced value | Dashboard refresh, infotainment, non-critical telemetry, map update |

The same technology can belong to different classes depending on use. A camera system used for emergency braking can be hard real-time. The same camera used for recording trip footage is not hard real-time.

### Case Study - Autonomous Vehicle

Emergency braking is hard real-time because a delayed brake command may fail to prevent collision. Traffic-sign recognition can be firm real-time because a sign detected after the vehicle has passed it may be discarded. Infotainment display updates are soft real-time because delay reduces user experience but does not immediately create a safety failure.

### Case Study - Industrial Robot

A safety light curtain that stops a robot arm is hard real-time. A production counter display is soft real-time. A quality-inspection image that arrives after the product leaves the inspection point can be firm real-time.

## 3. Real-Time Tasks and Events

A real-time task is a unit of computation or communication that must satisfy timing constraints. In autonomous systems, tasks include sensor sampling, perception, planning, control, actuation, message transmission, encryption, IDS checking, and logging.

### Periodic Tasks

Periodic tasks execute repeatedly at fixed intervals. Example: sample wheel speed every 10 ms. Periodic tasks are usually represented by:

```
Task_i = (C_i, T_i, D_i)
```

where `C_i` is computation time, `T_i` is period, and `D_i` is relative deadline.

### Aperiodic Tasks

Aperiodic tasks occur irregularly and have no guaranteed minimum separation. Example: a user requests a diagnostic report. Aperiodic tasks can create overload if not controlled.

### Sporadic Tasks

Sporadic tasks occur irregularly but have a known minimum inter-arrival time. Example: an emergency obstacle event may occur unpredictably but cannot occur more frequently than a specified physical or system limit.

### Time-Triggered and Event-Triggered Events

Time-triggered events are released by a predefined clock schedule. They are predictable and easier to analyze. Event-triggered events are released when a condition occurs, such as packet arrival, sensor threshold crossing, or obstacle detection. Event-triggered systems are responsive but require overload protection.

### Architecture

```
Clock Schedule -----> Time-Triggered Task ----                                             Scheduler -> Execution -> Deadline Check
External Event ----> Event-Triggered Task ----/
```

Good real-time design often combines both: periodic sensor sampling plus event-triggered emergency handling.

## 4. Timing Parameters

Timing analysis describes when work is released, started, executed, completed, and judged against a deadline.

| Parameter | Meaning |
|---|---|
| Event occurrence time | Time at which the real-world event happens |
| Release time | Time at which the task becomes ready |
| Arrival time | Time at which a job or packet enters a queue |
| Start time | Time at which execution begins |
| Execution time / computation time | Time spent executing on processor |
| Waiting time | Time spent ready but not executing |
| Completion / finish time | Time at which the job finishes |
| Response time | Time from release or arrival to completion |
| Turnaround time | Time from arrival to completion |

### Formulas

```
waiting_time = start_time - release_time
finish_time = start_time + execution_time
response_time = finish_time - release_time
turnaround_time = finish_time - arrival_time
```

If release time and arrival time are equal, response time and turnaround time are the same. In networked systems they may differ because a physical event can happen before a packet reaches the processing queue.

In [1]:
tasks = [
    {"name": "brake_control", "release": 0, "start": 1, "execution": 3, "deadline": 6},
    {"name": "object_tracking", "release": 0, "start": 3, "execution": 5, "deadline": 7},
    {"name": "dashboard_update", "release": 0, "start": 8, "execution": 2, "deadline": 9},
]

for task in tasks:
    finish = task["start"] + task["execution"]
    response = finish - task["release"]
    margin = task["deadline"] - finish
    status = "MEETS DEADLINE" if finish <= task["deadline"] else "MISSES DEADLINE"
    print(f'{task["name"]}: finish={finish}, response={response}, margin={margin}, {status}')

brake_control: finish=4, response=4, margin=2, MEETS DEADLINE
object_tracking: finish=8, response=8, margin=-1, MISSES DEADLINE
dashboard_update: finish=10, response=10, margin=-1, MISSES DEADLINE


## 5. Timing Constraints

A timing constraint defines when a task result must be available.

Relative deadline is measured from release time. Absolute deadline is measured on the system timeline:

```
absolute_deadline = release_time + relative_deadline
```

A deadline is met when:

```
finish_time <= absolute_deadline
```

A deadline miss occurs when:

```
finish_time > absolute_deadline
```

Deadline margin is:

```
deadline_margin = absolute_deadline - finish_time
```

Positive margin means the task completed before the deadline. Zero means it completed exactly at the deadline. Negative margin means a deadline miss.

Slack time and laxity estimate how much spare time remains before a deadline:

```
slack = absolute_deadline - current_time - remaining_execution_time
laxity = deadline - current_time - remaining_computation_time
```

Worst-Case Execution Time, or WCET, is the maximum execution time under defined assumptions. Best-Case Execution Time, or BCET, is the minimum execution time. Hard real-time systems must be analyzed with WCET, not only average execution time.

### Scheduling Formula Examples

For Rate Monotonic Scheduling:

```
U = sum(C_i / T_i)
RMS_bound = n * (2^(1/n) - 1)
```

For an ideal single-processor EDF model:

```
U = sum(C_i / T_i) <= 1
```

These formulas depend on modelling assumptions such as independent tasks, preemption, and known execution times.

In [2]:
import math

task_set = [
    {"name": "sensor_sample", "C": 1, "T": 5},
    {"name": "control_update", "C": 2, "T": 10},
    {"name": "telemetry", "C": 1, "T": 20},
]

utilization = sum(task["C"] / task["T"] for task in task_set)
n = len(task_set)
rms_bound = n * (2 ** (1 / n) - 1)

print(f"Total utilization = {utilization:.3f}")
print(f"RMS sufficient bound for {n} tasks = {rms_bound:.3f}")
print("RMS sufficient test:", "PASS" if utilization <= rms_bound else "INCONCLUSIVE")
print("Ideal EDF utilization test:", "PASS" if utilization <= 1 else "FAIL")

Total utilization = 0.450
RMS sufficient bound for 3 tasks = 0.780
RMS sufficient test: PASS
Ideal EDF utilization test: PASS


## 6. Communication Performance Parameters

Real-time networks must be evaluated using communication timing metrics.

Latency is the time taken for a packet or message to travel from sender to receiver:

```
latency = receive_time - send_time
```

Jitter is variation in latency:

```
jitter_i = abs(latency_i - latency_(i-1))
```

Throughput is useful delivered data per unit time:

```
throughput = delivered_bits / observation_time
```

Bandwidth is the nominal or available capacity of the channel. Throughput is actual delivered performance after protocol overhead, queueing, retransmission, contention, and loss.

Packet transmission time is:

```
transmission_time = packet_size_bits / link_rate_bits_per_second
```

End-to-end delay is:

```
end_to_end_delay = processing_delay + queueing_delay + transmission_delay + propagation_delay
```

Packet loss rate and reliability are:

```
packet_loss_rate = lost_packets / sent_packets
reliability = delivered_packets / sent_packets
```

Communication overhead includes headers, acknowledgements, encryption metadata, retransmission, routing, and synchronization.

In [3]:
send_times = [0.000, 0.010, 0.020, 0.030, 0.040]
receive_times = [0.008, 0.019, 0.027, 0.044, 0.051]
packet_size_bits = 1200 * 8
observation_time = receive_times[-1] - send_times[0]

latencies = [r - s for s, r in zip(send_times, receive_times)]
jitters = [abs(latencies[i] - latencies[i - 1]) for i in range(1, len(latencies))]
throughput = packet_size_bits * len(receive_times) / observation_time

print("Latencies in ms:", [round(x * 1000, 2) for x in latencies])
print("Jitter samples in ms:", [round(x * 1000, 2) for x in jitters])
print(f"Mean latency = {sum(latencies) / len(latencies) * 1000:.2f} ms")
print(f"Mean jitter = {sum(jitters) / len(jitters) * 1000:.2f} ms")
print(f"Throughput = {throughput / 1000:.2f} kbps")

Latencies in ms: [8.0, 9.0, 7.0, 14.0, 11.0]
Jitter samples in ms: [1.0, 2.0, 7.0, 3.0]
Mean latency = 9.80 ms
Mean jitter = 3.25 ms
Throughput = 941.18 kbps


## 7. Real-Time Communication Requirements

Real-time communication requires more than high bandwidth.

Bounded latency means the message delay has a known upper limit under defined conditions. Low jitter means delay does not vary too much from one message to another. Predictable communication means timing can be analyzed before deployment. Reliability means messages are delivered with acceptable success probability or recovery mechanisms. Availability means the communication system is usable when needed.

Deterministic message delivery can be supported by priority arbitration, time slots, traffic shaping, time synchronization, redundancy, admission control, and deadline-aware scheduling.

Deadline-aware communication prioritizes messages according to urgency and usefulness. A braking warning should not wait behind a large non-critical telemetry upload.

### Architecture

```
Message Generator
  -> Priority / Deadline Classification
  -> Real-Time Network Scheduler
  -> Transmission Medium
  -> Receiver Queue
  -> Deadline and Integrity Check
  -> Application Decision
```

Security checks also consume time. Encryption, hashing, authentication, signature verification, IDS inspection, and risk checks must be included in the timing budget.

## 8. Timing Analysis in Autonomous Systems

A typical autonomous timing chain is:

```
Sensor
  -> Perception
  -> Sensor Fusion
  -> Planning
  -> Control
  -> Network / Bus
  -> Actuator
  -> Physical Vehicle Response
```

The perception-to-action delay is:

```
perception_to_action_delay =
    sensor_capture_time
  + preprocessing_time
  + inference_time
  + planning_time
  + control_time
  + communication_time
  + actuator_response_time
```

For braking:

```
reaction_distance = vehicle_speed * total_system_delay
braking_distance = vehicle_speed^2 / (2 * deceleration)
stopping_distance = reaction_distance + braking_distance
```

Deadline verification:

```
system_is_timely = measured_response_time <= required_deadline
```

### Case Study - Autonomous Braking

A vehicle detects an obstacle 30 meters ahead while travelling at 20 m/s. If total system delay is high, the reaction distance grows before braking even begins. This shows why perception, control, communication, and actuation delays must be analyzed together.

### Case Study - V2X Hazard Warning

A vehicle broadcasts a hazard warning to nearby vehicles. The warning must arrive before the receiver needs to act. A correct warning that arrives late is not useful. The communication deadline depends on speed, distance, and reaction time.

### Case Study - Industrial Automation

A controller sends a stop command to a motor drive. If network jitter causes inconsistent command timing, the machine may not stop inside the safety envelope. Deterministic communication is required.

In [4]:
speed = 20.0               # m/s
obstacle_distance = 30.0   # m
system_delay = 0.180       # seconds
deceleration = 7.5         # m/s^2

reaction_distance = speed * system_delay
braking_distance = speed ** 2 / (2 * deceleration)
stopping_distance = reaction_distance + braking_distance
margin = obstacle_distance - stopping_distance

print(f"Reaction distance = {reaction_distance:.2f} m")
print(f"Braking distance = {braking_distance:.2f} m")
print(f"Stopping distance = {stopping_distance:.2f} m")
print(f"Safety margin = {margin:.2f} m")
print("Decision:", "timely under this simplified model" if margin >= 0 else "not timely under this simplified model")

Reaction distance = 3.60 m
Braking distance = 26.67 m
Stopping distance = 30.27 m
Safety margin = -0.27 m
Decision: not timely under this simplified model


## Detailed Case Studies and Formula Summary

### Case Study 1 - Sensor to Controller to Actuator in Autonomous Braking

Consider a vehicle travelling at high speed. A camera or radar detects an obstacle. The sensor data must be captured, transferred to the perception module, processed by the object-detection algorithm, passed to the planner, converted into a braking command, transmitted to the actuator controller, and finally applied by the brake system.

```
Obstacle Event
  -> Sensor Capture
  -> Perception Processing
  -> Planning and Control
  -> Vehicle Bus Communication
  -> Brake Actuator
  -> Physical Deceleration
```

The full timing chain is:

```
total_delay =
    sensing_delay
  + preprocessing_delay
  + inference_delay
  + planning_delay
  + bus_delay
  + actuator_delay
```

The deadline should be derived from the available stopping distance, not guessed. If the system is late, the vehicle may start braking after the safe response window has passed. This is a hard real-time case.

### Case Study 2 - V2X Cooperative Hazard Warning

A vehicle detects an accident or sudden braking event and broadcasts a warning. Nearby vehicles receive the message and decide whether to slow down. This case combines real-time communication and security.

```
Hazard Detection
  -> Message Creation
  -> Authentication / Integrity Protection
  -> Wireless Transmission
  -> Receiver Verification
  -> Driver or Vehicle Warning
```

The message must satisfy:

```
communication_delay + security_verification_delay <= warning_deadline
```

If cryptographic verification is skipped, spoofed messages may be accepted. If verification is too slow, correct messages may arrive too late. The correct design budgets time for both timing and security.

### Case Study 3 - Industrial Conveyor Sorting System

A product passes a sensor on a conveyor. The controller must classify the product and activate a diverter at the correct position. If the diverter signal is late, the product passes the actuation point.

```
Product Sensor
  -> Classification Task
  -> Controller Output
  -> Actuator Driver
  -> Diverter Movement
```

This may be firm or hard real-time depending on the consequence. If a missed item only reduces production quality, it may be firm. If the wrong item causes machine damage or safety risk, it becomes hard real-time.

### Consolidated Formula Sheet

Task timing:

```
finish_time = start_time + execution_time
waiting_time = start_time - release_time
response_time = finish_time - release_time
absolute_deadline = release_time + relative_deadline
deadline_margin = absolute_deadline - finish_time
```

Deadline decision:

```
deadline_met = finish_time <= absolute_deadline
deadline_missed = finish_time > absolute_deadline
```

Scheduling:

```
utilization = sum(C_i / T_i)
RMS_sufficient_bound = n * (2^(1/n) - 1)
EDF_ideal_uniprocessor_condition = utilization <= 1
```

Communication:

```
latency = receive_time - send_time
jitter_i = abs(latency_i - latency_(i-1))
throughput = delivered_bits / observation_time
packet_transmission_time = packet_size_bits / link_rate_bits_per_second
end_to_end_delay = processing + queueing + transmission + propagation
```

Reliability:

```
packet_loss_rate = lost_packets / sent_packets
delivery_ratio = delivered_packets / sent_packets
```

Vehicle timing:

```
reaction_distance = speed * total_delay
braking_distance = speed^2 / (2 * deceleration)
stopping_distance = reaction_distance + braking_distance
```

Security timing:

```
secure_message_delay = communication_delay + verification_delay
freshness_valid = message_timestamp within allowed_time_window
```

### How to Write CO 1 Answers

For definitions, include both correctness and timing. For classifications, always mention the consequence of lateness. For formulas, define each variable. For case studies, identify the event, task, deadline, timing chain, and failure consequence. For communication questions, compare latency, jitter, throughput, bandwidth, reliability, and deadline satisfaction rather than discussing only speed.

### Key Takeaways

The central idea of CO 1 is that time is part of correctness. A real-time design must identify deadlines, measure or bound execution and communication delay, classify the consequence of lateness, and verify whether the complete chain meets the timing requirement. In autonomous systems, this timing chain often crosses sensors, processors, networks, security modules, and actuators. A weak link anywhere in the chain can invalidate the final decision.

## Exam and Viva Questions

1. Why is temporal correctness as important as logical correctness in real-time systems?
2. Why is a fast system not automatically a real-time system?
3. Distinguish hard, firm, and soft real-time systems with autonomous-system examples.
4. Explain the difference between periodic, aperiodic, and sporadic tasks.
5. Define release time, start time, finish time, response time, and waiting time.
6. How are relative deadline and absolute deadline related?
7. What is slack time? Why is it useful?
8. Why is WCET more important than average execution time for hard real-time systems?
9. Distinguish bandwidth and throughput.
10. Why can jitter be dangerous in a control loop?
11. What is perception-to-action delay?
12. How does communication delay affect autonomous braking?

## References

- Python Software Foundation, `time` module documentation: https://docs.python.org/3/library/time.html
- C. L. Liu and J. W. Layland, scheduling theory for hard real-time environments: https://dl.acm.org/doi/10.1145/321738.321743
- IEEE 802.1 Time-Sensitive Networking working group: https://1.ieee802.org/tsn/
- NIST SP 800-30 Rev. 1 risk-assessment guidance: https://csrc.nist.gov/pubs/sp/800/30/r1/final
- NIST FIPS 180-4 Secure Hash Standard: https://csrc.nist.gov/pubs/fips/180-4/upd1/final